# Sleep-stage CNN classifier

This notebook follows the paper-inspired design: each 30-second target epoch uses five preceding, the target, and four following epochs (300 samples at 1 Hz). The input channels are Polar heart rate (1 Hz), CIN2, and CIN4 (20 Hz, resampled to 1 Hz). Apple Watch stages are used only as labels. One complete sleep is held out for validation.

The convolutional blocks use Conv1d, batch normalization, ReLU, max-pooling by 2, and dropout `p=0.1`. Training uses He initialization, Adam (`lr=1e-4`), batch size 100, a factor-2 learning-rate reduction after 10 epochs without validation-loss improvement, and early stopping after 30 epochs without improvement.

In [54]:
from __future__ import annotations

import copy
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, sosfiltfilt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATASET_DIR = Path.cwd()
if DATASET_DIR.name != "datasets_for_training":
    raise RuntimeError("Open and run this notebook with datasets_for_training as the working directory.")
VALIDATION_DATE = "2026-10-17"
SAMPLE_RATE = 20
FDC_HIGH_PASS_CUTOFF_HZ = 0.1
EPOCH_SECONDS = 30
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 4
CONTEXT_EPOCHS = CONTEXT_BEFORE + 1 + CONTEXT_AFTER
SAMPLES_PER_EPOCH = EPOCH_SECONDS * SAMPLE_RATE
INPUT_SAMPLES = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
BATCH_SIZE = 100
LEARNING_RATE = 1e-4
DROPOUT = 0.1
MAX_EPOCHS = 300
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 30
STAGES = ["Awake", "REM", "Core", "Deep"]
STAGE_TO_INDEX = {stage: index for index, stage in enumerate(STAGES)}
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required by this notebook, but torch.cuda.is_available() is False.")
DEVICE = torch.device("cuda")
print("Dataset:", DATASET_DIR)
print("Device:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__)
print("Validation sleep:", VALIDATION_DATE)

Dataset: c:\BME\Szemfedo\CapMeter\data_analysis\sleep_tests\datasets_for_training
Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch: 2.11.0+cu128
Validation sleep: 2026-10-17


## Loading and aligning the three sources

The trimmed files are treated as synchronized recordings: time zero is the first sample in each trimmed file. FDC timestamps are converted to seconds relative to their first row; Polar time is converted to seconds relative to its first retained sample; Apple Watch intervals are converted relative to its first interval. The common interval is represented at the native 20 Hz FDC rate. The 1 Hz heart-rate signal is linearly interpolated onto that 20 Hz grid.

In [55]:
def _date_from_name(path: Path) -> str:
    match = re.fullmatch(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match is None:
        raise ValueError(f"Unexpected Apple Watch filename: {path.name}")
    return match.group(1)


def _find_files(recording_date: str) -> tuple[Path, Path, Path]:
    apple = DATASET_DIR / f"{recording_date}.csv"
    fdc = DATASET_DIR / f"fdc1004_data_{recording_date.replace('-', '_')}.csv"
    polar_files = sorted(DATASET_DIR.glob(f"polar_{recording_date}_*.CSV"))
    if not apple.is_file() or not fdc.is_file() or not polar_files:
        raise FileNotFoundError(f"Missing one or more files for {recording_date}")
    return apple, polar_files[0], fdc


def _load_sleep(recording_date: str) -> pd.DataFrame:
    apple_path, polar_path, fdc_path = _find_files(recording_date)
    apple = pd.read_csv(apple_path)
    for column in ("Start time", "End time"):
        apple[column] = pd.to_datetime(apple[column]).map(
            lambda value: value.tz_localize(None) if value.tzinfo else value
        )
    apple_start = apple["Start time"].min()
    apple["start_s"] = (apple["Start time"] - apple_start).dt.total_seconds()
    apple["end_s"] = (apple["End time"] - apple_start).dt.total_seconds()

    polar = pd.read_csv(polar_path)
    polar["time_s"] = pd.to_timedelta(polar["Time"]).dt.total_seconds()
    polar["time_s"] -= polar["time_s"].min()
    polar["HR (bpm)"] = pd.to_numeric(polar["HR (bpm)"], errors="coerce")
    polar = polar.dropna(subset=["time_s", "HR (bpm)"]).drop_duplicates("time_s")

    fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])
    timestamp_ms = pd.to_numeric(fdc["timestamp_ms"], errors="coerce")
    fdc["time_s"] = (timestamp_ms - timestamp_ms.iloc[0]) / 1000
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = pd.to_numeric(fdc[column], errors="coerce")
    fdc = fdc.dropna(subset=["time_s", "CIN2_pF", "CIN4_pF"])
    fdc_filter = butter(
        N=4,
        Wn=FDC_HIGH_PASS_CUTOFF_HZ,
        btype="highpass",
        fs=SAMPLE_RATE,
        output="sos",
    )
    for column in ("CIN2_pF", "CIN4_pF"):
        fdc[column] = sosfiltfilt(fdc_filter, fdc[column].to_numpy(np.float64)).astype(np.float32)

    duration = min(float(apple["end_s"].max()), float(polar["time_s"].max()), float(fdc["time_s"].max()))
    samples = np.arange(0, math.floor(duration * SAMPLE_RATE) + 1, dtype=np.float32)
    seconds = samples / SAMPLE_RATE
    result = pd.DataFrame({"time_s": seconds})
    for column in ("CIN2_pF", "CIN4_pF"):
        result[column] = np.interp(seconds, fdc["time_s"], fdc[column])
    result["HR (bpm)"] = np.interp(seconds, polar["time_s"], polar["HR (bpm)"])
    result["stage"] = pd.NA
    for row in apple.itertuples():
        mask = (result["time_s"] >= row.start_s) & (result["time_s"] < row.end_s)
        result.loc[mask, "stage"] = row._1
    return result.dropna(subset=["stage"]).reset_index(drop=True)


recording_dates = sorted(_date_from_name(path) for path in DATASET_DIR.glob("20??-??-??.csv"))
if len(recording_dates) != 4:
    raise ValueError(f"Expected four training sleeps, found: {recording_dates}")
if VALIDATION_DATE not in recording_dates:
    raise ValueError(f"VALIDATION_DATE={VALIDATION_DATE} is not one of {recording_dates}")
sleep_data = {recording_date: _load_sleep(recording_date) for recording_date in recording_dates}
for recording_date, data in sleep_data.items():
    print(recording_date, len(data), "20-Hz rows", data["stage"].value_counts().to_dict())

C:\Users\Roli\AppData\Local\Temp\ipykernel_42952\1789997422.py:34: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  fdc = pd.read_csv(fdc_path, usecols=["timestamp_ms", "CIN2_pF", "CIN4_pF"])


2026-10-16 428881 20-Hz rows {'Core': 228381, 'REM': 107460, 'Deep': 78620, 'Awake': 14420}
2026-10-17 421861 20-Hz rows {'Core': 177621, 'Deep': 108020, 'REM': 103800, 'Awake': 32420}
2026-10-19 397347 20-Hz rows {'Core': 196240, 'Deep': 93020, 'REM': 89487, 'Awake': 18600}
2026-10-20 176700 20-Hz rows {'Core': 105100, 'Deep': 52820, 'REM': 18780}


## Epoch and context-window construction

Each label is the majority Apple Watch stage in the center 30-second epoch. Windows with missing or mixed center labels are excluded. The context is zero-padded only at the recording boundaries; because the first and last four/five epochs are not fully observed, the default construction below uses only complete 10-epoch windows.

In [56]:
def _make_examples(data: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    usable_samples = len(data) // SAMPLES_PER_EPOCH * SAMPLES_PER_EPOCH
    data = data.iloc[:usable_samples].copy()
    values = data[["HR (bpm)", "CIN2_pF", "CIN4_pF"]].to_numpy(np.float32)
    labels = data["stage"].map(STAGE_TO_INDEX).to_numpy()
    epoch_labels = []
    for start in range(0, usable_samples, SAMPLES_PER_EPOCH):
        block = labels[start:start + SAMPLES_PER_EPOCH]
        epoch_labels.append(int(np.bincount(block, minlength=len(STAGES)).argmax()))
    examples, targets = [], []
    for center in range(CONTEXT_BEFORE, len(epoch_labels) - CONTEXT_AFTER):
        start = (center - CONTEXT_BEFORE) * SAMPLES_PER_EPOCH
        stop = (center + CONTEXT_AFTER + 1) * SAMPLES_PER_EPOCH
        examples.append(values[start:stop])
        targets.append(epoch_labels[center])
    return np.asarray(examples, dtype=np.float32), np.asarray(targets, dtype=np.int64)


def _standardize(train_x: np.ndarray, other_x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std[std < 1e-6] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std


train_parts = [_make_examples(sleep_data[d]) for d in recording_dates if d != VALIDATION_DATE]
validation_x, validation_y = _make_examples(sleep_data[VALIDATION_DATE])
train_x = np.concatenate([part[0] for part in train_parts])
train_y = np.concatenate([part[1] for part in train_parts])
train_x, validation_x = _standardize(train_x, validation_x)
train_x = torch.from_numpy(train_x.transpose(0, 2, 1))
validation_x = torch.from_numpy(validation_x.transpose(0, 2, 1))
train_y = torch.from_numpy(train_y)
validation_y = torch.from_numpy(validation_y)
print("Train:", tuple(train_x.shape), tuple(train_y.shape))
print("Validation:", tuple(validation_x.shape), tuple(validation_y.shape))

Train: (1643, 3, 6000) (1643,)
Validation: (694, 3, 6000) (694,)


In [57]:
class SleepCNN(nn.Module):
    def __init__(self, channels: int = 3, classes: int = 4) -> None:
        super().__init__()
        blocks = []
        in_channels = channels
        for _ in range(6):
            blocks.extend([
                nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Dropout(DROPOUT),
            ])
            in_channels = 64
        self.features = nn.Sequential(*blocks)
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, channels, INPUT_SAMPLES)).flatten(1).shape[1]
        self.classifier = nn.Linear(feature_size, classes)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x).flatten(1))


model = SleepCNN().to(DEVICE)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
validation_loader = DataLoader(TensorDataset(validation_x, validation_y), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
print(model)

SleepCNN(
  (features): Sequential(
    (0): Conv1d(3, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.1, inplace=False)
    (5): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.1, inplace=False)
    (10): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (11): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.1, inplace=False)
    (15): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))

In [58]:
def _evaluate(loader: DataLoader) -> tuple[float, np.ndarray]:
    model.eval()
    total_loss, predictions, targets = 0.0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(features)
            total_loss += loss_function(logits, labels).item() * labels.size(0)
            predictions.append(logits.argmax(1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.column_stack((np.concatenate(targets), np.concatenate(predictions)))


history, best_state, best_loss, stale = [], None, float("inf"), 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for features, labels in train_loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss_function(model(features), labels).backward()
        optimizer.step()
    validation_loss, _ = _evaluate(validation_loader)
    scheduler.step(validation_loss)
    history.append((epoch, validation_loss, optimizer.param_groups[0]["lr"]))
    if validation_loss < best_loss - 1e-6:
        best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
    else:
        stale += 1
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch={epoch:03d} validation_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e}")
    if stale >= EARLY_STOPPING_PATIENCE:
        print("Early stopping at epoch", epoch)
        break
model.load_state_dict(best_state)
print("Best validation loss:", best_loss)

epoch=001 validation_loss=1.40246 lr=1.00e-04
epoch=010 validation_loss=1.22997 lr=1.00e-04
epoch=020 validation_loss=1.27167 lr=5.00e-05
epoch=030 validation_loss=1.36527 lr=2.50e-05
Early stopping at epoch 36
Best validation loss: 1.206362045120437


In [59]:
def classification_report(targets: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    matrix = np.zeros((len(STAGES), len(STAGES)), dtype=np.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    rows = []
    for index, stage in enumerate(STAGES):
        tp = matrix[index, index]
        precision = tp / matrix[:, index].sum() if matrix[:, index].sum() else 0.0
        recall = tp / matrix[index].sum() if matrix[index].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"stage": stage, "precision": precision, "recall": recall, "f1": f1, "support": matrix[index].sum()})
    total = matrix.sum()
    accuracy = np.trace(matrix) / total if total else 0.0
    expected = (matrix.sum(axis=0) * matrix.sum(axis=1)).sum() / total**2 if total else 0.0
    kappa = (accuracy - expected) / (1 - expected) if expected < 1 else 0.0
    report = pd.DataFrame(rows).set_index("stage")
    print("Accuracy:", accuracy, "Cohen kappa:", kappa)
    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(matrix, index=STAGES, columns=STAGES))
    return report


validation_loss, result = _evaluate(validation_loader)
report = classification_report(result[:, 0], result[:, 1])
display(report)
print("TST ground truth (hours):", (result[:, 0] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)
print("TST CNN (hours):", (result[:, 1] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)

Accuracy: 0.47406340057636887 Cohen kappa: 0.22066135026705064
Confusion matrix (rows=true, columns=predicted):


,Awake,REM,Core,Deep
Awake,0,16,38,0
REM,0,64,80,29
Core,0,49,172,66
Deep,0,68,19,93


,precision,recall,f1,support
stage,,,,
Awake,0.000000,0.000000,0.000000,54
REM,0.324873,0.369942,0.345946,173
Core,0.556634,0.599303,0.577181,287
Deep,0.494681,0.516667,0.505435,180


TST ground truth (hours): 5.333333333333333
TST CNN (hours): 5.783333333333333
